In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Workloads taken from the assignment PDF
ACTIVITY_1 = [
    {"Process": "P1", "Task": "Water-quality monitoring", "Arrival": 0, "Burst": 5},
    {"Process": "P2", "Task": "Waste-collection scheduling", "Arrival": 1, "Burst": 3},
    {"Process": "P3", "Task": "Traffic-pollution analysis", "Arrival": 2, "Burst": 4},
    {"Process": "P4", "Task": "Public energy-use monitoring", "Arrival": 3, "Burst": 2},
]
ACTIVITY_2 = [
    {"Process": "P1", "Task": "Flood-warning analysis", "Arrival": 0, "Burst": 5},
    {"Process": "P2", "Task": "Air-quality alert generation", "Arrival": 1, "Burst": 4},
    {"Process": "P3", "Task": "Emergency evacuation-route update", "Arrival": 2, "Burst": 3},
    {"Process": "P4", "Task": "Renewable-grid status monitoring", "Arrival": 3, "Burst": 2},
]

def simulate_rr(processes, quantum, overhead=0):
    """Round Robin simulation. Context overhead is added only between different processes."""
    procs = [dict(p) for p in processes]
    remaining = {p["Process"]: p["Burst"] for p in procs}
    first_start, completion = {}, {}
    ready = []
    segments = []
    time = 0
    i = 0
    last_process = None
    context_switches = 0
    total_overhead = 0

    # Add processes that have arrived at current time.
    while i < len(procs) and procs[i]["Arrival"] <= time:
        ready.append(procs[i]["Process"])
        i += 1

    while len(completion) < len(procs):
        if not ready:
            # CPU idle until the next process arrives
            next_arrival = procs[i]["Arrival"]
            if time < next_arrival:
                segments.append({"Start": time, "End": next_arrival, "Label": "IDLE", "Type": "Idle"})
                time = next_arrival
            while i < len(procs) and procs[i]["Arrival"] <= time:
                ready.append(procs[i]["Process"])
                i += 1
            continue

        current = ready.pop(0)

        # Add context switching overhead only when changing from one process to another.
        if last_process is not None and current != last_process and overhead > 0:
            context_switches += 1
            segments.append({"Start": time, "End": time + overhead, "Label": "CS", "Type": "Context switch"})
            time += overhead
            total_overhead += overhead
            while i < len(procs) and procs[i]["Arrival"] <= time:
                ready.append(procs[i]["Process"])
                i += 1
        elif last_process is not None and current != last_process:
            context_switches += 1

        if current not in first_start:
            first_start[current] = time

        run_for = min(quantum, remaining[current])
        start = time
        end = time + run_for
        segments.append({"Start": start, "End": end, "Label": current, "Type": "Process"})
        time = end
        remaining[current] -= run_for

        # Arrivals during this time slice enter the queue before the running process is requeued.
        while i < len(procs) and procs[i]["Arrival"] <= time:
            ready.append(procs[i]["Process"])
            i += 1

        if remaining[current] > 0:
            ready.append(current)
        else:
            completion[current] = time

        last_process = current

    rows = []
    for p in procs:
        name = p["Process"]
        ct = completion[name]
        tat = ct - p["Arrival"]
        wt = tat - p["Burst"]
        rt = first_start[name] - p["Arrival"]
        rows.append({
            "Process": name, "Sustainability task": p["Task"],
            "Arrival (ms)": p["Arrival"], "Burst (ms)": p["Burst"],
            "Completion (ms)": ct, "Turnaround (ms)": tat,
            "Waiting (ms)": wt, "Response (ms)": rt
        })
    metrics = pd.DataFrame(rows)
    total_elapsed = max(s["End"] for s in segments) if segments else 0
    busy = sum(s["End"] - s["Start"] for s in segments if s["Type"] == "Process")
    cpu_efficiency = (busy / total_elapsed * 100) if total_elapsed else 0
    summary = {
        "Average turnaround (ms)": metrics["Turnaround (ms)"].mean(),
        "Average waiting (ms)": metrics["Waiting (ms)"].mean(),
        "Average response (ms)": metrics["Response (ms)"].mean(),
        "Context switches": context_switches,
        "Total context-switch overhead (ms)": total_overhead,
        "CPU efficiency (%)": cpu_efficiency,
        "Total elapsed time (ms)": total_elapsed,
    }
    return segments, metrics, summary

def draw_gantt(segments, title):
    fig, ax = plt.subplots(figsize=(13, 2.8))
    labels = {"P1": "P1", "P2": "P2", "P3": "P3", "P4": "P4",
              "CS": "Context switch", "IDLE": "CPU idle"}
    process_labels = list(dict.fromkeys(s["Label"] for s in segments))
    cmap = plt.get_cmap("tab10")
    color_map = {name: cmap(i % 10) for i, name in enumerate(process_labels)}
    for s in segments:
        start, end, label = s["Start"], s["End"], s["Label"]
        ax.barh(0, end-start, left=start, height=0.55,
                color=color_map[label], edgecolor="black", alpha=0.88)
        ax.text((start+end)/2, 0, f"{label}\n{start}-{end}",
                ha="center", va="center", fontsize=8, color="black")
    ax.set_yticks([])
    ax.set_xlabel("Time (milliseconds)")
    ax.set_title(title)
    ax.set_xticks(range(0, max(s["End"] for s in segments)+1))
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()

def inspect_time(segments, t):
    # Each interval is [Start, End); report what runs during millisecond t.
    for s in segments:
        if s["Start"] <= t < s["End"]:
            return s
    return None

activity = widgets.ToggleButtons(
    options=[("Activity 1: No overhead", "a1"),
             ("Activity 2: 1 ms overhead", "a2")],
    description="Workload:"
)
quantum = widgets.IntSlider(value=2, min=1, max=5, step=1,
                            description="Time quantum (ms)", continuous_update=False)
show_comparison = widgets.Checkbox(value=True, description="Show quantum comparison")
output = widgets.Output()

def update_visualizer(*args):
    with output:
        clear_output(wait=True)
        if activity.value == "a1":
            workload, overhead = ACTIVITY_1, 0
            comparison_quanta = [2, 3]
            title = "Activity 1 — Round Robin without context-switch overhead"
        else:
            workload, overhead = ACTIVITY_2, 1
            comparison_quanta = [2, 4]
            title = "Activity 2 — Round Robin with 1 ms context-switch overhead"

        segments, metrics, summary = simulate_rr(workload, quantum.value, overhead)
        print(title)
        print(f"Selected time quantum: {quantum.value} ms | Context-switch overhead: {overhead} ms")
        draw_gantt(segments, f"{title} (Quantum = {quantum.value} ms)")

        print("PROCESS PERFORMANCE")
        display(metrics.round(2).style.format(precision=2))

        print("OVERALL PERFORMANCE")
        display(pd.DataFrame([summary]).round(2).style.format(precision=2))

        print("MILLISECOND-BY-MILLISECOND TIMELINE")
        timeline_rows = []
        for t in range(int(summary["Total elapsed time (ms)"])):
            s = inspect_time(segments, t)
            if s:
                timeline_rows.append({
                    "Time interval (ms)": f"{t}–{t+1}",
                    "What happens": s["Label"] if s["Type"] == "Process" else s["Type"],
                    "Segment interval (ms)": f'{s["Start"]}–{s["End"]}'
                })
        timeline = pd.DataFrame(timeline_rows)
        time_slider = widgets.IntSlider(
            value=0, min=0, max=max(0, int(summary["Total elapsed time (ms)"])-1),
            step=1, description="Inspect ms:", continuous_update=False
        )
        detail = widgets.Output()

        def update_detail(change=None):
            with detail:
                clear_output(wait=True)
                t = time_slider.value
                s = inspect_time(segments, t)
                if s:
                    if s["Type"] == "Process":
                        print(f"At {t}–{t+1} ms: {s['Label']} is using the CPU.")
                    elif s["Type"] == "Context switch":
                        print(f"At {t}–{t+1} ms: Context-switch overhead; no process executes.")
                    else:
                        print(f"At {t}–{t+1} ms: CPU is idle, waiting for a process.")
                    print(f"Current schedule segment: {s['Label']} from {s['Start']} to {s['End']} ms.")
                else:
                    print("No scheduled activity at this time.")
        time_slider.observe(update_detail, names="value")
        display(time_slider, detail)
        update_detail()
        display(timeline)

        if show_comparison.value:
            print(f"COMPARISON: quantum {comparison_quanta[0]} ms vs {comparison_quanta[1]} ms")
            comparison_rows = []
            for q in comparison_quanta:
                _, m, sm = simulate_rr(workload, q, overhead)
                comparison_rows.append({
                    "Quantum (ms)": q,
                    "Avg turnaround (ms)": sm["Average turnaround (ms)"],
                    "Avg waiting (ms)": sm["Average waiting (ms)"],
                    "Avg response (ms)": sm["Average response (ms)"],
                    "Context switches": sm["Context switches"],
                    "Overhead (ms)": sm["Total context-switch overhead (ms)"],
                    "CPU efficiency (%)": sm["CPU efficiency (%)"]
                })
            display(pd.DataFrame(comparison_rows).round(2).style.format(precision=2))
            print("Use the comparison to discuss fairness, waiting/response time and switching overhead.")

activity.observe(update_visualizer, names="value")
quantum.observe(update_visualizer, names="value")
show_comparison.observe(update_visualizer, names="value")
display(widgets.VBox([activity, quantum, show_comparison, output]))
update_visualizer()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')